# 01 — Why Agents Are Hard to Test

## Why this notebook exists

You know how to build agents — the `langgraph/`, `mcp/`, and `a2a/` series in this repo cover wiring them together and making them communicate. But once an agent is running, a new set of questions arrives: *Does it actually work? How do I know it hasn't regressed? What happens when the output changes phrasing but is still semantically right?*

Traditional unit tests answer questions like "does `2 + 2` return `4`?" — the answer is deterministic, there is exactly one correct form, and every step is independent. Agents violate all three assumptions. This notebook makes that concrete before we introduce any tooling: we'll build a tiny toy agent, write the most natural test for it, watch it flake, and understand *why*. That understanding is the foundation everything else in this series builds on.

No API key. No external packages. Just Python.

## What you'll learn

- Why a naive `assert agent(input) == "expected"` unit test breaks for agents even when the agent is *correct*.
- The three root causes that make agent evaluation hard: **non-determinism**, **no single ground truth**, and **multi-step compounding failures**.
- How to quantify flakiness concretely: loop over seeds, count pass vs fail on semantically-correct outputs.
- Why we need **graders** — functions that *score* an output — instead of equality checks, and what the grader shape `(example, output) -> Score` looks like (preview only; the harness is built in notebooks 02–03).

## 1. Setup

This notebook uses only the Python standard library: `random` for the stub agent's phrasing variation and `dataclasses` for a preview of the `Score` type in the closing section. No `pip install` needed.

One important note on seeds: the stub agent in this notebook uses a **per-call seeded `random.Random` instance** whose seed is derived from the call number, not from global state. That means:
- Running any single call multiple times gives the same result — it is reproducible.
- Running across a *range* of call numbers gives genuinely different phrasings — it looks non-deterministic.

This is intentional: it lets us demonstrate flakiness in a completely reproducible way.

In [1]:
import random
from dataclasses import dataclass

# We will use this throughout the notebook.
# No external dependencies — stdlib only.

print("Python stdlib only. No pip install needed.")
print(f"random module: {random.__name__}")
print(f"dataclasses module: {dataclass.__module__}")

Python stdlib only. No pip install needed.
random module: random
dataclasses module: dataclasses


## 2. A Toy Sentiment-Classifier Agent

Our toy agent is a sentiment classifier: given a review string, it decides whether the sentiment is positive or negative. A real LLM would return something like `"positive"` — but it wouldn't always phrase it the *same* way. Sometimes it would say `"Positive!"`, sometimes `"The sentiment is positive."`, sometimes `"positive"`, sometimes `"POSITIVE"`.

We simulate that variability with a seeded `random.Random` instance. The agent always gets the *answer* right (positive reviews return a positive-sentiment string, negative reviews return a negative-sentiment string), but the *phrasing* varies. This is the critical distinction: **semantic correctness ≠ string equality.**

The `call_count` parameter lets us drive different phrasing choices without touching global state, so every output below is fully reproducible.

In [2]:
# Phrasings the stub agent chooses among for each sentiment.
# All positive phrasings are semantically equivalent; same for negative.
_POSITIVE_PHRASINGS = [
    "positive",
    "Positive",
    "Positive!",
    "POSITIVE",
    "The sentiment is positive.",
    "sentiment: positive",
    "positive sentiment",
]

_NEGATIVE_PHRASINGS = [
    "negative",
    "Negative",
    "Negative!",
    "NEGATIVE",
    "The sentiment is negative.",
    "sentiment: negative",
    "negative sentiment",
]

# Positive signal words. A real LLM would do inference; we just keyword-match
# for simplicity — the agent is always "correct" in that positive reviews get
# a positive phrasing and negative reviews get a negative phrasing.
_POSITIVE_SIGNALS = {"great", "love", "excellent", "amazing", "good", "fantastic",
                     "wonderful", "best", "happy", "perfect"}


def sentiment_agent(review: str, call_count: int = 0) -> str:
    """Toy sentiment classifier that always gives the right ANSWER
    but varies its PHRASING based on call_count.

    Args:
        review: The product review text to classify.
        call_count: Drives phrasing variation. Different values → different
                    surface strings, even for the same review.

    Returns:
        A string expressing the sentiment — phrasing varies.
    """
    words = set(review.lower().split())
    is_positive = bool(words & _POSITIVE_SIGNALS)

    # Seed a fresh Random instance from call_count so output is reproducible
    # but varies across different call_count values.
    rng = random.Random(call_count)

    if is_positive:
        return rng.choice(_POSITIVE_PHRASINGS)
    else:
        return rng.choice(_NEGATIVE_PHRASINGS)


print("Agent defined. Classifies sentiment; phrasing varies by call_count.")

Agent defined. Classifies sentiment; phrasing varies by call_count.


In [3]:
### Try it

POSITIVE_REVIEW = "I love this product, it is great and amazing!"
NEGATIVE_REVIEW = "This is the worst thing I have ever bought."

print("Same positive review, different call_count values:")
for i in range(7):
    output = sentiment_agent(POSITIVE_REVIEW, call_count=i)
    print(f"  call_count={i}  ->  {output!r}")

print()
print("Same negative review, different call_count values:")
for i in range(7):
    output = sentiment_agent(NEGATIVE_REVIEW, call_count=i)
    print(f"  call_count={i}  ->  {output!r}")

Same positive review, different call_count values:
  call_count=0  ->  'positive sentiment'
  call_count=1  ->  'Positive'
  call_count=2  ->  'positive sentiment'
  call_count=3  ->  'Positive'
  call_count=4  ->  'Positive'
  call_count=5  ->  'The sentiment is positive.'
  call_count=6  ->  'positive sentiment'

Same negative review, different call_count values:
  call_count=0  ->  'negative sentiment'
  call_count=1  ->  'Negative'
  call_count=2  ->  'negative sentiment'
  call_count=3  ->  'Negative'
  call_count=4  ->  'Negative'
  call_count=5  ->  'The sentiment is negative.'
  call_count=6  ->  'negative sentiment'


## 3. The Naive Test — and Why It Flakes

The most natural test for our agent is:

```python
assert sentiment_agent(POSITIVE_REVIEW) == "positive"
```

This is exactly how we'd test a deterministic function. The test only passes for the calls where the agent happens to return the bare string `"positive"` — which calls those are depends on the Python version's random stream, so the pass/fail pattern itself is unreliable. A *semantically correct* output like `"Positive!"` or `"positive sentiment"` still causes the assertion to fail.

Let's make the flakiness concrete: run the same assertion across all 7 phrasing seeds and count how many pass.

> **Gotcha:** A test that passes most of the time is worse than a test that always fails — it creates false confidence and surfaces failures at random moments, making it hard to distinguish a real regression from natural variation.

In [4]:
NUM_SEEDS = 7  # One per phrasing in _POSITIVE_PHRASINGS

passed = 0
failed = 0

print(f"Running: assert sentiment_agent(POSITIVE_REVIEW, call_count=i) == 'positive'")
print(f"across {NUM_SEEDS} seeds (one per distinct phrasing):\n")

for i in range(NUM_SEEDS):
    output = sentiment_agent(POSITIVE_REVIEW, call_count=i)
    try:
        assert output == "positive", f"Got {output!r}, expected 'positive'"
        passed += 1
        print(f"  seed {i}: PASS  (output={output!r})")
    except AssertionError as e:
        failed += 1
        print(f"  seed {i}: FAIL  ({e})")

print()
print(f"Results: {passed}/{NUM_SEEDS} passed, {failed}/{NUM_SEEDS} failed")
print()
print("Every FAIL is a semantically correct output that the naive test rejects.")
print("The agent is not wrong — the test is.")

Running: assert sentiment_agent(POSITIVE_REVIEW, call_count=i) == 'positive'
across 7 seeds (one per distinct phrasing):

  seed 0: FAIL  (Got 'positive sentiment', expected 'positive')
  seed 1: FAIL  (Got 'Positive', expected 'positive')
  seed 2: FAIL  (Got 'positive sentiment', expected 'positive')
  seed 3: FAIL  (Got 'Positive', expected 'positive')
  seed 4: FAIL  (Got 'Positive', expected 'positive')
  seed 5: FAIL  (Got 'The sentiment is positive.', expected 'positive')
  seed 6: FAIL  (Got 'positive sentiment', expected 'positive')

Results: 0/7 passed, 7/7 failed

Every FAIL is a semantically correct output that the naive test rejects.
The agent is not wrong — the test is.


## 4. The Three Core Difficulties

The flaky test above is a symptom of three deeper problems. Each has its own runnable demo below.

### 4a. Non-Determinism — Same Input, Different Surface Text

A real LLM's temperature parameter, sampling order, and context window all affect the exact tokens produced. Our stub models this with `call_count`. The *meaning* is stable; the *string* is not. Any test that compares strings will therefore be fragile.

The cell below calls the agent 10 times on the same input, collects all unique surface strings, and prints the semantic verdict for each. Same input, same correct answer, many distinct strings.

In [5]:
# 4a: Non-determinism demo
review = "This product is excellent and I love it!"

outputs = [sentiment_agent(review, call_count=i) for i in range(10)]
unique_outputs = sorted(set(outputs))  # sort for stable display

print(f"Input: {review!r}")
print(f"10 calls → {len(unique_outputs)} distinct surface strings:\n")
for s in unique_outputs:
    print(f"  {s!r}")

print()
print("All are semantically 'positive', yet string equality only passes when the output is exactly the bare string 'positive'.")
print(f"String equality test would pass {outputs.count('positive')}/10 times.")

Input: 'This product is excellent and I love it!'
10 calls → 5 distinct surface strings:

  'POSITIVE'
  'Positive'
  'Positive!'
  'The sentiment is positive.'
  'positive sentiment'

All are semantically 'positive', yet string equality only passes when the output is exactly the bare string 'positive'.
String equality test would pass 0/10 times.


### 4b. No Single Ground Truth — Many Correct Outputs

For tasks like summarization, explanation, or translation there is no single canonical correct output. Two summaries can both be excellent and completely non-identical. Storing one as the "golden answer" and asserting equality would reject all the other good answers.

The cell below constructs two clearly-good one-sentence summaries of the same article, shows they are not equal, and shows that neither is a substring of the other — so `in`-tests also fail.

In [6]:
# 4b: No single ground truth demo
ARTICLE = (
    "Researchers at Stanford have developed a new battery technology "
    "that charges in under five minutes and lasts three times longer "
    "than current lithium-ion cells, potentially transforming electric vehicles."
)

# Two equally-valid one-sentence summaries a capable LLM might produce.
SUMMARY_A = (
    "Stanford researchers developed a fast-charging battery that lasts "
    "three times longer than lithium-ion, which could revolutionize EVs."
)
SUMMARY_B = (
    "A new Stanford battery technology charges in under five minutes "
    "and offers triple the longevity of lithium-ion cells."
)

print("Article:")
print(f"  {ARTICLE}\n")
print("Summary A:")
print(f"  {SUMMARY_A}\n")
print("Summary B:")
print(f"  {SUMMARY_B}\n")

print(f"Are they equal?             {SUMMARY_A == SUMMARY_B}")
print(f"Is B a substring of A?      {SUMMARY_B in SUMMARY_A}")
print(f"Is A a substring of B?      {SUMMARY_A in SUMMARY_B}")
print()
print("Both summaries are accurate and high quality.")
print("No equality or substring test can accept both and reject bad ones.")

Article:
  Researchers at Stanford have developed a new battery technology that charges in under five minutes and lasts three times longer than current lithium-ion cells, potentially transforming electric vehicles.

Summary A:
  Stanford researchers developed a fast-charging battery that lasts three times longer than lithium-ion, which could revolutionize EVs.

Summary B:
  A new Stanford battery technology charges in under five minutes and offers triple the longevity of lithium-ion cells.

Are they equal?             False
Is B a substring of A?      False
Is A a substring of B?      False

Both summaries are accurate and high quality.
No equality or substring test can accept both and reject bad ones.


### 4c. Multi-Step Compounding — One Variation Breaks the Next Step

Real agents chain steps: an LLM extracts information, then downstream code parses it, then another LLM uses the parsed result. If step 1's output varies in phrasing, step 2's brittle parser may break — even though step 1 was semantically correct. The error compounds: by the time something visibly goes wrong, the failure is two steps removed from its actual cause.

The cell below models a two-step pipeline:
- **Step 1 (agent):** classifies sentiment — output phrasing varies.
- **Step 2 (parser):** extracts a boolean `is_positive` from step 1's output via a brittle exact-string check.

Step 2 works for one phrasing and silently produces the wrong boolean for all others.

In [7]:
# 4c: Multi-step compounding demo

def step1_classify(review: str, call_count: int = 0) -> str:
    """Step 1: sentiment classification (phrasing varies)."""
    return sentiment_agent(review, call_count=call_count)


def step2_parse(classification: str) -> bool:
    """Step 2: brittle parser — only recognises the bare lowercase string."""
    # Real downstream code often looks like this after a quick prototype.
    return classification == "positive"


def pipeline(review: str, call_count: int = 0) -> dict:
    raw = step1_classify(review, call_count=call_count)
    is_positive = step2_parse(raw)
    return {"raw": raw, "is_positive": is_positive}


review = POSITIVE_REVIEW
print(f"Input: {review!r}\n")
print(f"{'call_count':<12} {'step1 output':<35} {'step2 result':<14} {'correct?'}")
print("-" * 75)

for i in range(7):
    result = pipeline(review, call_count=i)
    correct = result["is_positive"]  # should always be True for a positive review
    marker = "OK" if correct else "BUG <-- compounding failure"
    print(f"{i:<12} {result['raw']:<35} {str(result['is_positive']):<14} {marker}")

print()
print("Step 1 is always semantically correct.")
bug_count = sum(1 for i in range(7) if not pipeline(review, call_count=i)["is_positive"])
print(f"Step 2 breaks silently on {bug_count}/7 phrasings — the bug is invisible at step 1.")

Input: 'I love this product, it is great and amazing!'

call_count   step1 output                        step2 result   correct?
---------------------------------------------------------------------------
0            positive sentiment                  False          BUG <-- compounding failure
1            Positive                            False          BUG <-- compounding failure
2            positive sentiment                  False          BUG <-- compounding failure
3            Positive                            False          BUG <-- compounding failure
4            Positive                            False          BUG <-- compounding failure
5            The sentiment is positive.          False          BUG <-- compounding failure
6            positive sentiment                  False          BUG <-- compounding failure

Step 1 is always semantically correct.
Step 2 breaks silently on 7/7 phrasings — the bug is invisible at step 1.


## 5. What We Actually Need — Graders, Not Matchers

The three demos above share a common shape: *we know what a good output looks like, but we can't express that knowledge as a string equality check.* What we need instead is a **grader** — a function that takes an output and returns a *score* rather than a boolean pass/fail.

A grader for the sentiment classifier might look like:

```python
def sentiment_grader(example, output) -> Score:
    label = output.lower()
    passed = example["expected_label"] in label   # "positive" anywhere in output
    return Score(
        key="sentiment_correct",
        score=1.0 if passed else 0.0,
        passed=passed,
        comment=f"output={output!r}",
    )
```

Notice the shape: `grader(example, output) -> Score`. The `Score` dataclass (introduced in notebook 02) holds:
- `key` — which grader produced this score.
- `score` — a float in `[0, 1]`.
- `passed` — a boolean threshold judgment.
- `comment` — optional explanation.

And `example` (introduced in notebook 03) is an `{input, expected, metadata}` bundle — one row from your eval dataset.

We're not implementing these yet. We're just naming the shapes so the rest of the series can refer back to this moment: the point where we realized `assert equals` wasn't enough and decided to build something better.

## What you just learned

- A toy sentiment agent can always produce the *right answer* while failing a naive `assert output == "positive"` test — because the output phrasing varies even when the semantics are correct.
- **Non-determinism:** the same input can produce many distinct surface strings; equality tests are fragile.
- **No single ground truth:** for open-ended outputs like summaries, many correct forms exist; no single golden string covers them.
- **Multi-step compounding:** a brittle parser in step 2 can silently produce wrong results whenever step 1's phrasing varies, making bugs hard to locate.
- We need **graders** — functions of shape `(example, output) -> Score` — that *score* outputs rather than match them, so semantically-correct variation doesn't register as failure.

## What's missing

We've named the problem and sketched the grader shape, but we haven't built any graders yet. The `Score` dataclass is undefined, `sentiment_grader` above is pseudocode, and we have no way to run a grader across a batch of examples.

**Notebook 02 — `02_assertions_and_golden_outputs.ipynb`** builds the first real graders: exact match, `contains`/regex match, and structured-output validation with `pydantic`. Each returns a proper `Score`. By the end of notebook 02 you'll have a small toolkit of deterministic graders that handle most of the cases where `assert equals` fails — no LLM required.